In [1]:
# Configuration

# zero-shot, few-shot, chain-of-thought, tree-of-thought
prompting_strategy = "chain-of-thought"
# openai/gpt-5.2, z-ai/glm-5, anthropic/claude-opus-4.5, openai/gpt-oss-120b, z-ai/glm-4.7-flash
model = "anthropic/claude-opus-4.5"
debug = True
root_dir = "../.."

In [2]:
ddl_prompt_path = f"{root_dir}/prompts/{prompting_strategy}/ddl.md"
dml_dql_prompt_path = f"{root_dir}/prompts/{prompting_strategy}/dml-dql.md"
ddl_prompt_template = None
dml_dql_prompt_template = None

with open(ddl_prompt_path) as file:
    ddl_prompt_template = file.read()
with open(dml_dql_prompt_path) as file:
    dml_dql_prompt_template = file.read()

In [3]:
import pandas as pd
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [4]:
ground_truth_df = pd.read_csv(f"{root_dir}/datasets/validation-set.csv")
ground_truth_df = ground_truth_df.drop(columns=['Comment'])
ground_truth_df.sort_values(by=["Antipattern", "Project", "File", "Line from"], ascending=True, ignore_index=True)

,Project,Antipattern,File,Line from,Line to
0,APSfurizon/fz-backend,31 Flavors,jooq-common/src/main/java/net/furizon/jooq/generated/tables/ExchangeConfirmationStatus.java,115,115
1,APSfurizon/fz-backend,31 Flavors,jooq-common/src/main/java/net/furizon/jooq/generated/tables/Media.java,90,90
2,APSfurizon/fz-backend,31 Flavors,jooq-common/src/main/java/net/furizon/jooq/generated/tables/MembershipInfo.java,168,168
3,APSfurizon/fz-backend,31 Flavors,jooq-common/src/main/java/net/furizon/jooq/generated/tables/MembershipInfo.java,183,183
4,APSfurizon/fz-backend,31 Flavors,jooq-common/src/main/java/net/furizon/jooq/generated/tables/MembershipInfo.java,198,198
5,APSfurizon/fz-backend,31 Flavors,jooq-common/src/main/java/net/furizon/jooq/generated/tables/Orders.java,93,93
6,APSfurizon/fz-backend,31 Flavors,jooq-common/src/main/java/net/furizon/jooq/generated/tables/Orders.java,135,135
7,APSfurizon/fz-backend,31 Flavors,jooq-common/src/main/java/net/furizon/jooq/generated/tables/Orders.java,175,175
8,APSfurizon/fz-backend,31 Flavors,jooq-common/src/main/java/net/furizon/jooq/generated/tables/Orders.java,240,240
9,DiSSCo/dissco-core-annotation-processing-service,31 Flavors,src/main/java/eu/dissco/annotationprocessingservice/database/jooq/tables/Annotation.java,133,133


In [5]:
projects = ground_truth_df["Project"].unique().tolist()
projects

['APSfurizon/fz-backend',
 'BluBluBlaBla/vidsummize',
 'boogiespook/waltz-test',
 'catsloveme/telegram-bot',
 'devstronomy/nasa-data-scraper',
 'DiSSCo/dissco-core-annotation-processing-service',
 'dndanoff/spring-boot-unit-integration-test-example',
 'ebi-gene-expression-group/annotare2',
 'hpple/revolut-backend-test',
 'jiangtj/jc-platform',
 'juanhaugaard/H2-sakila',
 'Macbeth-Klm/Tinkoff-java-course-2024',
 'MahjongRepository/YagiMotel',
 'puzink/Wallet',
 'ralscha/attic',
 'ralscha/blog2020',
 'SleepwalkerPasha/egorov_pa_project',
 'tapis-project/tapis-notifications',
 'therepanic/lapayment-crypto-pay-system',
 'wuhunyu/jooq-transaction']

In [6]:
# Excluding files generated based on system schemas and database extensions/migration tools
excluded_path_fragments = [
    # Java
    'module-info.java',
    'package-info.java',
    
    # MySQL
    'information_schema/InformationSchema.java',
    'information_schema/tables',
    'mysql/Mysql.java',
    'mysql/tables',
    'performance_schema/PerformanceSchema.java',
    'performance_schema/tables',
    'sys/Sys.java',
    'sys/tables',

    # Flyway
    'FlywaySchemaHistory.java',

    # Liquibase
    'Databasechangelog.java',
    'Databasechangeloglock.java',

    # PostgreSQL
    'information_schema/Domains.java',
    'pg_catalog/PgCatalog.java',
    'pg_catalog/tables',

    # Quartz
    'tables/Qrtz',

    # PGP
    'PgpArmorHeaders.java',

    # Dblink
    'tables/Dblink',

    # hstore
    'tables/Each.java',
    'tables/Skeys.java',
    'tables/Svals.java'
]

# Including files that contain a reference to jOOQ or a potential reference to a generated file (the latter may include false positives)
included_phrases = [
    'org.jooq',
    'Tables',
    'tables'
]

# Excluding generated classes, which contain redundant information that doesn't help with detection
excluded_phrases = [
    'extends TableRecordImpl',
    'extends UpdatableRecordImpl',
    'extends org.jooq.impl.UpdatableRecordImpl',
    'extends CatalogImpl',
    'extends SchemaImpl',
    'extends UDTImpl',
    'extends UDTRecordImpl',
    'extends DAOImpl',
    'extends AbstractRoutine',
    'implements EnumType',
    'implements DeserializableJooqEnum',
    'public class Indexes',
    'public class Keys',
    'public class Routines',
    'public class Sequences',
    'public class Tables',
    'public class Domains',
    'class AbstractSpringDAOImpl',
    'extends AbstractSpringDAOImpl',
    'tables.pojos;',
    'tables.interfaces;',
    'Targeted by JavaCPP'
]

ddl_indicator_phrases = [
    'This class is generated by jOOQ',
    'extends TableImpl',
    'extends org.jooq.impl.TableImpl'
]

In [7]:
import glob, os

_keys_cache = {}

def get_project_dir(project: str):
    return f"{root_dir}/repositories/{project.replace("/", "_")}"


def find_all_java_file_paths(project: str):
    return glob.glob(get_project_dir(project) + "/**/*.java", recursive=True)

def find_applicable_file_paths(project: str):
    applicable_file_paths = []
    for file_path in find_all_java_file_paths(project):
        if all(fragment not in file_path for fragment in excluded_path_fragments) and os.path.isfile(file_path):
            with open(file_path, encoding='latin-1') as file:
                file_contents = file.read()
                if any(phrase in file_contents for phrase in included_phrases) and all(phrase not in file_contents for phrase in excluded_phrases):
                    applicable_file_paths.append(file_path)
    return applicable_file_paths


def find_keys_file_paths(project: str):
    # Check if we have already scanned this project for keys
    if project in _keys_cache:
        return _keys_cache[project]

    keys_file_paths = []
    for file_path in find_all_java_file_paths(project):
        if os.path.isfile(file_path):
            try:
                with open(file_path, encoding="latin-1") as file:
                    # Reading only the first few KB is often enough for class declarations
                    # but for safety, we search the whole file once.
                    if "public class Keys " in file.read():
                        keys_file_paths.append(file_path)
            except Exception:
                continue
    
    _keys_cache[project] = keys_file_paths
    return keys_file_paths

def find_closest_keys_file(project: str, file_path: str):
    closest_keys_file_path = None
    closest_common_prefix_length = 0

    for keys_file_path in find_keys_file_paths(project):
        common_prefix_length = len(os.path.commonprefix([file_path, keys_file_path]))
        if common_prefix_length > closest_common_prefix_length:
            closest_keys_file_path = keys_file_path
            closest_common_prefix_length = common_prefix_length
        
    return closest_keys_file_path


def prepare_prompt(project: str, file_path: str):
    with open(file_path) as file:
        lines = file.readlines()
    with open(find_closest_keys_file(project, file_path)) as file:
        keys_lines = file.readlines()

    numbered_contents = "\n".join(
        f"{idx + 1}: {line.strip()}" for idx, line in enumerate(lines)
    )
    keys_file_contents = "\n".join(line.strip() for line in keys_lines)

    is_ddl = any(phrase in numbered_contents for phrase in ddl_indicator_phrases)
    prompt_template = ddl_prompt_template if is_ddl else dml_dql_prompt_template
    prompt_template = prompt_template.replace('FILE_CONTENTS', numbered_contents)
    prompt_template = prompt_template.replace('KEYS_CONTENTS', keys_file_contents)
    return prompt_template

In [8]:
from openai import AsyncOpenAI
import asyncio
from pydantic import BaseModel
from enum import Enum
import json
import time

client = AsyncOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

class AntipatternName(str, Enum):
    id_required = 'ID Required',
    keyless_entry = 'Keyless Entry',
    rounding_errors = 'Rounding Errors',
    thirty_one_flavors = '31 Flavors',
    poor_mans_search_engine = 'Poor Man’s Search Engine',
    implicit_columns = 'Implicit Columns',
    beware_of_the_unknown = 'Beware of the Unknown'

class AntipatternOccurrence(BaseModel):
    antipatternName: AntipatternName
    linesRangeStart: int
    linesRangeEnd: int
    codeFragment: str
    reasoning: str

class Response(BaseModel):
    chainOfThought: list[str]
    occurrences: list[AntipatternOccurrence]

progress_lock = asyncio.Lock()
processed_count = 0
total_files = sum(len(list(find_applicable_file_paths(p))) for p in projects)
global_input_tokens = 0
global_output_tokens = 0
global_total_tokens = 0
global_total_cost = 0.0
global_retry_count = 0

CONCURRENCY_LIMIT = 10
semaphore = asyncio.Semaphore(CONCURRENCY_LIMIT)

async def generate(project: str, file_path: str):
    global processed_count, global_input_tokens, global_output_tokens, global_total_tokens, global_total_cost, global_retry_count

    while True:
        async with semaphore:
            try:
                prompt = prepare_prompt(project, file_path)
                response = await client.responses.parse(
                    model=model,
                    reasoning={"effort": "none"},
                    temperature=0.0,
                    text_format=Response,
                    input=[{"role": "user", "content": prompt}],
                )

                async with progress_lock:
                    global_input_tokens += response.usage.input_tokens
                    global_output_tokens += response.usage.output_tokens
                    global_total_tokens += response.usage.total_tokens
                    global_total_cost += response.usage.cost
        
                if response.output_parsed == None:
                    async with progress_lock:
                        global_retry_count += 1
                    print(f"Received empty response: {file_path}")
                    print(response)
                    continue
                    
                async with progress_lock:
                    processed_count += 1
                    print(f"[{processed_count}/{total_files}] File analyzed successfully: {file_path}")
        
                return {
                    "project": project,
                    "file": file_path,
                    "response": response.output_parsed,
                    "usage": response.usage,
                    "status": "success"
                }
            except Exception as e:
                async with progress_lock:
                    global_retry_count += 1
                print(f"Exception: {file_path}")
                print(e)

tasks = []

for project in projects:
    for file_path in find_applicable_file_paths(project):
        tasks.append(generate(project, file_path))

start_time = time.perf_counter()
results = await asyncio.gather(*tasks)
end_time = time.perf_counter()

total_runtime = end_time - start_time
success_count = sum(1 for res in results if res["status"] == "success")
error_count = total_files - success_count

print("\n--- Analysis Summary ---")
print(f"Total Runtime:      {total_runtime:.2f} seconds")
print(f"Files Processed:    {success_count} success, {error_count} failed")
print(f"Total Retries:      {global_retry_count}")
print(f"Input Tokens:       {global_input_tokens:,}")
print(f"Output Tokens:      {global_output_tokens:,}")
print(f"Total Tokens:       {global_total_tokens:,}")
print(f"Total Cost:         ${global_total_cost:.6f}")

[1/1159] File analyzed successfully: ../../repositories/APSfurizon_fz-backend/application/src/main/java/net/furizon/backend/infrastructure/jackson/JsonSerializer.java
[2/1159] File analyzed successfully: ../../repositories/APSfurizon_fz-backend/application/src/main/java/net/furizon/backend/infrastructure/jackson/SimpleJsonSerializer.java
[3/1159] File analyzed successfully: ../../repositories/APSfurizon_fz-backend/application/src/main/java/net/furizon/backend/infrastructure/security/SecurityUtils.java
[4/1159] File analyzed successfully: ../../repositories/APSfurizon_fz-backend/application/src/main/java/net/furizon/backend/infrastructure/configuration/DatabaseConfiguration.java
[5/1159] File analyzed successfully: ../../repositories/APSfurizon_fz-backend/application/src/main/java/net/furizon/backend/infrastructure/security/permissions/mapper/JooqUserHasRoleMapper.java
[6/1159] File analyzed successfully: ../../repositories/APSfurizon_fz-backend/application/src/main/java/net/furizon/bac

In [12]:
results_df = pd.DataFrame(columns = ["Project", "Antipattern", "File", "Line from", "Line to"])

for result in results:
    if result['status'] == "success":
        for occurrence in result['response'].occurrences:
            results_df.loc[len(results_df)] = {
                "Project": result["project"],
                "Antipattern": occurrence.antipatternName.value,
                "File": result["file"][(len(get_project_dir(result["project"])) + 1):],
                "Line from": occurrence.linesRangeStart,
                "Line to": occurrence.linesRangeEnd
            }
    else:
        print(f"Error: {result['error']}")

results_df.sort_values(by=["Antipattern", "File", "Line from"], ascending=True, ignore_index=True)

,Project,Antipattern,File,Line from,Line to
0,APSfurizon/fz-backend,31 Flavors,jooq-common/src/main/java/net/furizon/jooq/generated/tables/MembershipInfo.java,168,168
1,APSfurizon/fz-backend,31 Flavors,jooq-common/src/main/java/net/furizon/jooq/generated/tables/MembershipInfo.java,183,183
2,APSfurizon/fz-backend,31 Flavors,jooq-common/src/main/java/net/furizon/jooq/generated/tables/MembershipInfo.java,198,198
3,juanhaugaard/H2-sakila,31 Flavors,jooq-generated-sakila/src/main/java/org/tayrona/sakila/data/tables/Film.java,126,126
4,DiSSCo/dissco-core-annotation-processing-service,31 Flavors,src/main/java/eu/dissco/annotationprocessingservice/database/jooq/tables/Annotation.java,133,133
5,DiSSCo/dissco-core-annotation-processing-service,31 Flavors,src/main/java/eu/dissco/annotationprocessingservice/database/jooq/tables/MasJobRecord.java,65,65
6,DiSSCo/dissco-core-annotation-processing-service,31 Flavors,src/main/java/eu/dissco/annotationprocessingservice/database/jooq/tables/MasJobRecord.java,100,100
7,DiSSCo/dissco-core-annotation-processing-service,31 Flavors,src/main/java/eu/dissco/annotationprocessingservice/database/jooq/tables/MasJobRecord.java,110,110
8,ralscha/attic,31 Flavors,travellog/server/src/main/java/ch/rasc/travellog/db/tables/AppUser.java,87,88
9,APSfurizon/fz-backend,Beware of the Unknown,jooq-common/src/main/java/net/furizon/jooq/generated/tables/Authentications.java,88,88


In [13]:
import pandas as pd
import numpy as np

# --- 1. The Fixed Helper Functions ---

def iou_1d_inclusive(gt, pred):
    """
    Calculates Intersection over Union for DISCRETE inclusive intervals (lines).
    Math: Length = end - start + 1
    """
    # gt and pred are tuples/lists: (start, end)
    
    # 1. Calculate Intersection
    start_inter = max(gt[0], pred[0])
    end_inter = min(gt[1], pred[1])
    
    # We add +1 here because lines are discrete. 
    # If start=79, end=79, overlap is 1 line.
    inter_len = max(0, end_inter - start_inter + 1)
    
    # 2. Calculate Union
    # Union = Area_A + Area_B - Intersection
    len_gt = gt[1] - gt[0] + 1
    len_pred = pred[1] - pred[0] + 1
    
    union_len = len_gt + len_pred - inter_len
    
    return inter_len / union_len if union_len > 0 else 0.0

def calculate_counts(gt_intervals, pred_intervals, threshold=0.5):
    """
    Returns (True Positives for Recall, True Positives for Precision)
    """
    # 1. Calculate Recall Matches (Did we find the GT?)
    tp_recall_count = 0
    for g in gt_intervals:
        # If this GT overlaps sufficiently with ANY prediction
        if any(iou_1d_inclusive(g, p) >= threshold for p in pred_intervals):
            tp_recall_count += 1
            
    # 2. Calculate Precision Matches (Was this Prediction correct?)
    tp_precision_count = 0
    for p in pred_intervals:
        # If this Prediction overlaps sufficiently with ANY GT
        if any(iou_1d_inclusive(g, p) >= threshold for g in gt_intervals):
            tp_precision_count += 1
            
    return tp_recall_count, tp_precision_count

# --- 2. Data Preparation ---

# Ensure strict types
ground_truth_df['Line from'] = ground_truth_df['Line from'].astype(int)
ground_truth_df['Line to'] = ground_truth_df['Line to'].astype(int)
results_df['Line from'] = results_df['Line from'].astype(int)
results_df['Line to'] = results_df['Line to'].astype(int)

# Strip whitespace from string columns just in case
ground_truth_df['File'] = ground_truth_df['File'].str.strip()
results_df['File'] = results_df['File'].str.strip()
ground_truth_df['Project'] = ground_truth_df['Project'].str.strip()
results_df['Project'] = results_df['Project'].str.strip()

# --- 3. Metric Calculation ---

all_antipatterns = set(ground_truth_df['Antipattern']).union(set(results_df['Antipattern']))
metric_rows = []

for ap in all_antipatterns:
    # Filter Dataframes by Antipattern first
    gt_subset = ground_truth_df[ground_truth_df['Antipattern'] == ap]
    pred_subset = results_df[results_df['Antipattern'] == ap]
    
    # Get all unique (Project, File) combinations
    # We use tuples to ensure we match strictly on both
    gt_keys = set(zip(gt_subset['Project'], gt_subset['File']))
    pred_keys = set(zip(pred_subset['Project'], pred_subset['File']))
    relevant_keys = gt_keys.union(pred_keys)
    
    ap_tp_recall = 0   
    ap_tp_precision = 0 
    ap_total_gt = 0    
    ap_total_pred = 0  
    
    for (proj, filename) in relevant_keys:
        # Extract intervals where BOTH Project and File match
        gts = gt_subset[
            (gt_subset['Project'] == proj) & 
            (gt_subset['File'] == filename)
        ][['Line from', 'Line to']].values.tolist()
        
        preds = pred_subset[
            (pred_subset['Project'] == proj) & 
            (pred_subset['File'] == filename)
        ][['Line from', 'Line to']].values.tolist()
        
        ap_total_gt += len(gts)
        ap_total_pred += len(preds)
        
        # Calculate Matches using the Inclusive IoU
        matches_rec, matches_prec = calculate_counts(gts, preds, threshold=0.5)
        
        ap_tp_recall += matches_rec
        ap_tp_precision += matches_prec

    # Calculate Rates
    precision = ap_tp_precision / ap_total_pred if ap_total_pred > 0 else 0.0
    recall = ap_tp_recall / ap_total_gt if ap_total_gt > 0 else 0.0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
        
    metric_rows.append({
        "Antipattern": ap,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1,
        "Support (GT Count)": ap_total_gt,
        "Pred Count": ap_total_pred
    })

# --- 4. Weighted Average and Output ---

metrics_df = pd.DataFrame(metric_rows)
total_support = metrics_df["Support (GT Count)"].sum()

if total_support > 0:
    w_avg_prec = (metrics_df["Precision"] * metrics_df["Support (GT Count)"]).sum() / total_support
    w_avg_rec = (metrics_df["Recall"] * metrics_df["Support (GT Count)"]).sum() / total_support
    w_avg_f1 = (metrics_df["F1-Score"] * metrics_df["Support (GT Count)"]).sum() / total_support
else:
    w_avg_prec, w_avg_rec, w_avg_f1 = 0.0, 0.0, 0.0

metrics_df.loc[len(metrics_df)] = {
    "Antipattern": "WEIGHTED AVERAGE",
    "Precision": w_avg_prec,
    "Recall": w_avg_rec,
    "F1-Score": w_avg_f1,
    "Support (GT Count)": total_support,
    "Pred Count": metrics_df["Pred Count"].sum()
}

# Display
metrics_df.round(4)

,Antipattern,Precision,Recall,F1-Score,Support (GT Count),Pred Count
0,Implicit Columns,0.9650,0.9486,0.9567,350,343
1,Rounding Errors,1.0000,1.0000,1.0000,10,10
2,Poor Man’s Search Engine,0.5789,0.8000,0.6718,15,19
3,Keyless Entry,0.7237,0.9821,0.8333,56,76
4,31 Flavors,1.0000,0.6000,0.7500,15,9
5,ID Required,0.9479,0.9192,0.9333,99,96
6,Beware of the Unknown,0.8750,0.3889,0.5385,36,16
7,WEIGHTED AVERAGE,0.9248,0.9002,0.9030,581,569


In [14]:
results

[{'project': 'APSfurizon/fz-backend',
  'file': '../../repositories/APSfurizon_fz-backend/application/src/main/java/net/furizon/backend/infrastructure/jackson/SimpleJsonSerializer.java',
  'response': Response(chainOfThought=['I need to analyze this Java class for SQL query antipatterns.', "Looking at the class, it's a `SimpleJsonSerializer` that implements `JsonSerializer`.", "The class uses Jackson's `ObjectMapper` to serialize objects to JSON strings.", 'It has two methods: `serializeAsString` and `serializeAsJson`.', 'The `serializeAsJson` method uses `JSON.valueOf()` which is a jOOQ JSON type, but this is just creating a JSON value, not querying the database.', 'This class does not contain any jOOQ DSL queries or DAO interactions.', 'There are no SELECT statements, no LIKE/ILIKE patterns, no column selections, and no NULL handling in query logic.', 'This is purely a utility class for JSON serialization, not database interaction.', 'No antipatterns are present in this code.'], occu

In [ ]:
Date	Slug	Usage	BYOK Usage	Requests	Prompt Tokens	Completion Tokens	Reasoning Tokens
2026-03-07 12:00:00	anthropic/claude-opus-4.5	40.862095	0	1159	5961739	442136	0
